# Kitsune Colab Workflow

This notebook modernizes Mirsky's Kitsune workflow so you can upload a capture file and inspect when attacks likely happened without setting up an old local environment first. Run the cells from top to bottom.


## 1. Install runtime dependencies

Why this cell matters: Kitsune needs `tshark` plus a few Python libraries to parse packet captures and compute anomaly scores inside Colab.


In [ ]:
# Install packet parsing and plotting dependencies used by the notebook.
!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y tshark
!pip install -q scapy cython numpy scipy matplotlib
!tshark --version | head -n 1


## 2. Load the project code

Why this cell matters: it pulls the current GitHub version of the project into Colab so the notebook uses the same code that is stored in the repository.


In [ ]:
# Replace this URL with your own repository if needed before running.
!git clone https://github.com/AXIOM-JYOTHESH/kitsune-py-colab.git
%cd /content/kitsune-py-colab


## 3. Prepare the Python environment

Why this cell matters: it makes the cloned repository importable, which lets the notebook call the original Kitsune implementation under the hood.


In [ ]:
import os
import sys
import numpy as np

# Point Python to the cloned repo so local modules can be imported.
sys.path.insert(0, '/content/kitsune-py-colab')


## 4. Choose the capture file and create the detector

Why this cell matters: this is where you connect your uploaded `.pcap` or `.pcapng` file to the Kitsune pipeline and define the detection settings.


In [ ]:
from Kitsune import Kitsune

# Update this path after uploading your packet capture to Colab.
capture_path = '/content/ARP_MitM_pcap.pcapng'
maxAE = 10
FMgrace = 5000
ADgrace = 50000
packet_limit = np.inf

# Initialize Kitsune on the selected capture file.
K = Kitsune(capture_path, packet_limit, maxAE, FMgrace, ADgrace)


## 5. Estimate processing time

Why this cell matters: it runs a small packet sample first so you can estimate how long the full analysis may take on a larger capture.


In [ ]:
import time

start = time.time()
test_packets = 10000

# Benchmark a short run before processing the full capture.
for i in range(test_packets):
    rmse = K.proc_next_packet()
    if rmse == -1:
        break

elapsed = time.time() - start
print("Seconds for 10,000 packets:", elapsed)
print("Estimated total hours:", (elapsed * 2504268 / test_packets) / 3600)


## 6. Run the full anomaly detection pass

Why this cell matters: this is the main analysis stage. Kitsune processes each packet and records an RMSE anomaly score that can reveal suspicious behavior.


In [ ]:
rmse_scores = []
i = 0

# Process every packet and store the anomaly score for later inspection.
while True:
    i += 1
    if i % 1000 == 0:
        print(f'Processed {i} packets')
    rmse = K.proc_next_packet()
    if rmse == -1:
        break
    rmse_scores.append(rmse)

print('Packets processed:', len(rmse_scores))


## 7. Visualize where attacks may have happened

Why this cell matters: spikes in the RMSE curve highlight unusual traffic regions, helping you inspect when attacks or anomalies likely occurred in the uploaded capture.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.plot(rmse_scores)
plt.yscale('log')
plt.title('Kitsune RMSE Scores')
plt.xlabel('Packet index')
plt.ylabel('RMSE (log scale)')
plt.show()
